# From model to decision {#sec-model-to-decision}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c11-model-to-decision.ipynb)

A hospital group has a waiting list of about 70,000 patients, half of them waiting more than a year. The board has money for one or two measures and asks three questions: what works, how sure are we, and how long until the longest waits are gone? A model earns its keep only when it answers questions like these in a form the board can act on.

```
                 ageing           ageing            ageing
Referrals --> [0 to 3] -------> [3 to 6] -------> [6 to 12] -------> [over 12]
               |    |            |    |            |    |             |    |
               v    v            v    v            v    v             v    v
          Treating  Validating  (each band is treated in proportion to its
                                 size times a priority weight; validation
                                 removes patients who no longer need care)
```

*The policy model of this chapter: four stocks of patients by months waited, one inflow, ageing between the bands, and two outflows from every band.*

This chapter turns the waiting-list model into a policy laboratory. You will design policies as parameter settings, compare them on several measures at once, test whether the ranking survives the things you do not know, estimate how long a backlog takes to clear, and write the result as a one-page brief. You will also wrap the model in a function that could sit behind sliders, and check our engine by reproducing a well-known published result.

**Who it is for.** Anyone who has a model and must now say what to do. **Who can skip.** Readers who already run policy experiments, robustness grids and regret analysis may read sections 4, 5 and 7 only. **Prerequisites.** Chapters 1 to 9; the ageing chain (chapter 4, Delays, pipelines and memory) and weighted splits (chapter 5, Decisions inside the model) in particular. **Time.** About three hours.

**Learning outcomes.** By the end of this chapter you can:

1. **formulate** a policy as a change of parameters in a stock-and-flow model (section 2, Try it);
2. **compare** policies on several outcome measures over time, including cost (section 2, Try it);
3. **justify** a recommendation with a scenario grid and a regret table (section 3, Try it);
4. **estimate** the time to clear a backlog, and explain why a naive estimate recedes (section 4, Try it);
5. **write** a one-page brief for a decision maker from model runs (section 5, Try it);
6. **build** a function that a slider interface could call (section 6, Try it);
7. **replicate** a published model's qualitative result in your own implementation (section 7, Try it).

In [ ]:
# Run this cell first. In Colab it installs loopbricks (about ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 10
rng = np.random.default_rng(SEED)

DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") else \
    "https://raw.githubusercontent.com/waelrash1/loopbricks/main/notebooks/data/waiting_list.csv"
wl = pd.read_csv(DATA, parse_dates=["month"])

# Okabe-Ito colours, each with its own line style, so figures read in greyscale
OKABE_ITO = ["#000000", "#E69F00", "#56B4E9", "#009E73", "#D55E00",
             "#0072B2", "#CC79A7"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = cycler(color=OKABE_ITO) + \
    cycler(linestyle=STYLES)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

## 1. A policy laboratory for the waiting list

The data are the synthetic monthly waiting list of the running case (synthetic, shaped like published Irish waiting-list counts; see chapter 10, Data, calibration and uncertainty). The last month, December 2023, is where the policy model starts.

In [ ]:
last = wl.iloc[-1]
print("Start of the policy runs:", last["month"].strftime("%B %Y"))
print(last[["total", "w0_3", "w3_6", "w6_12", "w12_plus"]].to_string())
change = wl["total"].diff()
print(f"Average monthly change in 2023: {change.iloc[-12:].mean():+.0f} "
      f"patients/month")

The list stood at 69,927 patients at the end of 2023, and 34,790 of them (about half) had waited more than 12 months. Through 2023 it grew by only about 16 patients a month: referrals and treatments were almost in balance.

The model follows the bands of the data, so it can speak about long waiters as well as the total. The four bands form an ageing chain, the **Ageing Chain** (`aging`) of chapter 4: patients move from one band to the next after 3, 3 and 6 months. Treatments are shared across the bands in proportion to the number in each band times a **priority weight**, the **Weighted Split** (`weightedSplit`) of chapter 5. Both are after Hines, *Molecules of Structure* [@hines_molecules], as is every molecule in this book. With weights of 1.4, 1.0, 0.9 and 0.8, the shorter waits are seen a little more often, as urgent cases are. The total stays the familiar rule of the case:

$$\text{Treatments} = \min\left(\text{Capacity},\ \frac{\text{Waiting list}}{\text{Minimum treatment time}}\right),$$

$$\text{Treating}_k = \text{Treatments}\times\frac{w_k\,B_k}{\sum_j w_j\,B_j},$$

where $B_k$ is the number waiting in band $k$ (patients) and $w_k$ its weight (dimensionless).

Referrals and capacity have no published series, so we need working values. Over 2023 the list grew by only about 16 patients a month, so treatments nearly matched referrals. We take `Referrals base` 5,400 patients/month growing 3% a year (a fraction of 0.03 per year) and `Capacity base` 5,380 patients/month growing 2% a year: close to where the data put referrals and treatments at the start of 2024 (chapter 10 shows how to estimate them). Section 3 asks how much these guesses matter.

Every policy below is a constant of the model, set to zero when the policy is off. That one design choice is what lets `run_batch` try every policy in every scenario in a single call.

In [ ]:
def stock(i, name, init, units="patients"):
    return {"id": i, "name": name, "init": str(init), "units": units,
            "nonNegative": True}


def flow(i, name, eq, src=None, dst=None, units="patients/month"):
    f = {"id": i, "name": name, "eq": eq, "units": units}
    if src:
        f["from"] = src
    if dst:
        f["to"] = dst
    return f


def var(i, name, eq, units=""):
    return {"id": i, "name": name, "eq": str(eq), "units": units}


BANDS = ["Waiting 0 to 3 months", "Waiting 3 to 6 months",
         "Waiting 6 to 12 months", "Waiting over 12 months"]
B = [b.replace(" ", "_") for b in BANDS]          # names in equations
COLS = ["w0_3", "w3_6", "w6_12", "w12_plus"]
ON = "STEP(1, Policy_start)"                      # 1 once a policy starts

policy_spec = {
    "name": "Waiting list policy model", "timeUnit": "month",
    "T": 60, "dt": 0.25,
    "stocks": [stock(f"b{k}", BANDS[k - 1], int(last[COLS[k - 1]]))
               for k in range(1, 5)]
    + [stock("cost", "Extra cost", 0, "EUR million")],
    "flows": [
        flow("ref", "Referrals", "Referral_trend * (1 - Demand_reduction"
             " * MIN(1, MAX(0, TIME - Policy_start) / 12))", dst="b1"),
        flow("a1", "Ageing 1", f"{B[0]} / 3", "b1", "b2"),
        flow("a2", "Ageing 2", f"{B[1]} / 3", "b2", "b3"),
        flow("a3", "Ageing 3", f"{B[2]} / 6", "b3", "b4"),
    ]
    + [flow(f"t{k}", f"Treating {k}",
            f"Treatments * Weight_{k} * {B[k - 1]} / Weighted_list",
            f"b{k}") for k in range(1, 5)]
    + [flow(f"v{k}", f"Validating {k}", f"{B[k - 1]} * Validation_rate",
            f"b{k}") for k in range(1, 5)]
    + [flow("spend", "Spending",
            "(Extra_capacity * Cost_permanent + Initiative_capacity"
            " * Cost_initiative + Validation_contacts * Cost_contact)"
            f" / 1000000 + Demand_programme_cost * {ON}"
            " * (Demand_reduction > 0)", dst="cost",
            units="EUR million/month")],
    "vars": [
        var("wl", "Waiting list", " + ".join(B), "patients"),
        var("rb", "Referrals base", 5400, "patients/month"),
        var("rg", "Referral growth", 0.03, "fraction/year"),
        var("rt", "Referral trend",
            "Referrals_base * (1 + Referral_growth) ^ (TIME / 12)",
            "patients/month"),
        var("cb", "Capacity base", 5380, "patients/month"),
        var("cg", "Capacity growth", 0.02, "fraction/year"),
        var("ct", "Capacity trend",
            "Capacity_base * (1 + Capacity_growth) ^ (TIME / 12)",
            "patients/month"),
        var("ps", "Policy start", 3, "month"),
        var("pe", "Permanent extra", 0, "fraction"),
        var("te", "Temporary extra", 0, "fraction"),
        var("td", "Temporary duration", 12, "month"),
        var("ec", "Extra capacity",
            f"Capacity_trend * Permanent_extra * {ON}", "patients/month"),
        var("ic", "Initiative capacity",
            "Capacity_trend * Temporary_extra * (STEP(1, Policy_start)"
            " - STEP(1, Policy_start + Temporary_duration))",
            "patients/month"),
        var("cap", "Capacity",
            "Capacity_trend + Extra_capacity + Initiative_capacity",
            "patients/month"),
        var("mtt", "Minimum treatment time", 0.5, "month"),
        var("tr", "Treatments",
            "MIN(Capacity, Waiting_list / Minimum_treatment_time)",
            "patients/month"),
        var("w1", "Weight 1", 1.4), var("w2", "Weight 2", 1.0),
        var("w3", "Weight 3", 0.9), var("w4", "Weight 4", 0.8),
        var("wsum", "Weighted list",
            " + ".join(f"Weight_{k} * {B[k - 1]}" for k in range(1, 5)),
            "patients"),
        var("dr", "Demand reduction", 0, "fraction"),
        var("dpc", "Demand programme cost", 0.3, "EUR million/month"),
        var("vf", "Validation fraction", 0, "fraction"),
        var("vd", "Validation duration", 6, "month"),
        var("vr", "Validation rate",
            "Validation_fraction / Validation_duration"
            " * (STEP(1, Policy_start)"
            " - STEP(1, Policy_start + Validation_duration))", "1/month"),
        var("vc", "Validation contacts",
            "Waiting_list / Validation_duration * (Validation_rate > 0)",
            "patients/month"),
        var("cpe", "Cost permanent", 1100, "EUR/patient"),
        var("cin", "Cost initiative", 1800, "EUR/patient"),
        var("cco", "Cost contact", 25, "EUR/patient"),
    ],
}
pm = lb.Model.from_spec(policy_spec)
print(pm)

The model has five stocks: the four bands and a running total of extra spending in EUR million. The costs are illustrative round numbers, not tariffs: a permanently funded treatment slot costs EUR 1,100 per patient, a treatment bought in a short initiative EUR 1,800 (premium rates and private providers), and a validation call or letter EUR 25 per patient on the list. Replace them with your own before any real use.

Before trying policies, run the model with every policy off, and draw it after the data. Time 0 is the end of December 2023 and the run goes 60 months, to the end of 2028.

In [ ]:
def year(t):
    # model time in months from the end of 2023 -> calendar year
    return 2023 + 11 / 12 + np.asarray(t) / 12


hist = wl["month"].dt.year + (wl["month"].dt.month - 1) / 12
base = pm.run()

fig, ax = plt.subplots()
ax.plot(hist, wl["total"] / 1000, label="Waiting list, data")
ax.plot(hist, wl["w12_plus"] / 1000, label="Over 12 months, data")
ax.plot(year(base.index), base["Waiting list"] / 1000,
        label="Waiting list, model")
ax.plot(year(base.index), base["Waiting over 12 months"] / 1000,
        label="Over 12 months, model")
ax.set(title="If nothing changes", xlabel="year",
       ylabel="patients waiting (thousand)")
ax.legend()
plt.show()
print(base.loc[[0, 12, 60], ["Waiting list", "Waiting over 12 months",
                             "Referrals", "Treatments"]].round(0))

*Figure: data from 2014 to 2023 and the model with no policy from 2024 to 2028. The total list and the number waiting more than 12 months, in thousands of patients.*

With no policy, referrals start just 20 patients/month above treatments, so the list barely moves at first. But referrals grow at 3% a year and capacity at 2%, so the gap widens: by month 60, 6,260 referrals a month meet 5,940 treatments, the list reaches 79,735 and the number waiting over 12 months 38,780. The projection continues the slow drift of the data rather than its jump in 2020, which came from the pandemic shock, not from the trend.

**Try it 1.**

- *Check.* Without running anything: in the first month, is the list growing or falling, and by about how much? Then check against the table.
- *Apply.* Halve `minimum_treatment_time` to 0.25 month. Does the no-policy projection change? Why, or why not?
- *Extend.* Our policy model starts its bands at the December 2023 counts. Start instead with every band scaled so the list totals 60,000 patients. How much of the difference at month 60 is left?

<details><summary>Worked answer</summary>

*Check.* Growing, by 20 patients/month: referrals of 5,400 against treatments of 5,380 patients/month.

*Apply.* No change: the list ends at 79,735 either way. `Waiting_list / Minimum_treatment_time` is about 140,000 patients/month, far above capacity, so the `MIN` always picks capacity. The minimum treatment time matters only for a list so short it could be cleared within weeks.

*Extend.* The list at month 60 is 69,808 against 79,735: the whole starting difference of 9,927 patients is still there. While capacity binds, treatments do not depend on the list, so the list is a pure accumulator and remembers where it started for ever. That is why a one-off reduction (validation, an initiative) shifts the path but not its slope.

</details>

In [ ]:
print("Month 0: referrals - treatments =",
      round(base.loc[0, "Referrals"] - base.loc[0, "Treatments"], 1))
quick = pm.run(minimum_treatment_time=0.25)
print("Halved minimum treatment time, list at month 60:",
      round(quick["Waiting list"].iloc[-1]))
m60 = pm
for k in range(4):
    m60 = m60.with_init(BANDS[k], last[COLS[k]] * 60000 / last["total"])
small = m60.run()
print("Start at 60,000: list at month 60",
      round(small["Waiting list"].iloc[-1]), "against",
      round(base["Waiting list"].iloc[-1]))

## 2. Policies as parameter settings, judged on several measures

A **policy** in a model is a change in a decision rule or in a constant that someone controls. Six are on the board's table:

| Policy | What changes in the model | Setting |
|---|---|---|
| More capacity, permanent | `Permanent extra` | 5% more capacity from month 3, for good |
| Waiting-list initiative | `Temporary extra` | 20% more capacity for 12 months, then back |
| Demand management | `Demand reduction` | referrals 4% lower, phased in over 12 months, EUR 0.3 million a month for the programme |
| Validate the list | `Validation fraction` | contact everyone over 6 months; about 10% no longer need care |
| Treat the longest waiters first | `Weight 4` | weight of the over-12 band from 0.8 to 2.0 |
| A package | all three of permanent, validation, targeting | as above |

The initiative and the permanent increase buy roughly the same number of extra treatments over five years (12 months at 20% against 57 months at 5% of capacity), so comparing them is fair. Each policy is a dictionary of settings.

In [ ]:
POLICIES = {
    "Do nothing": {},
    "Permanent +5%": {"permanent_extra": 0.05},
    "Initiative +20%, 1 year": {"temporary_extra": 0.20},
    "Demand management": {"demand_reduction": 0.04},
    "Validate the list": {"validation_fraction": 0.10},
    "Longest waiters first": {"weight_4": 2.0},
    "Package": {"permanent_extra": 0.05, "validation_fraction": 0.10,
                "weight_4": 2.0},
}
runs = {name: pm.run(**p) for name, p in POLICIES.items()}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for name, d in runs.items():
    x = year(d.index)
    axes[0].plot(x, d["Waiting list"] / 1000, label=name)
    axes[1].plot(x, d["Waiting over 12 months"] / 1000)
    axes[2].plot(x, d["Extra cost"])
axes[0].set(title="Waiting list", ylabel="patients (thousand)")
axes[1].set(title="Waiting over 12 months", ylabel="patients (thousand)")
axes[2].set(title="Extra spending, cumulative", ylabel="EUR million")
for a in axes:
    a.set_xlabel("year")
fig.legend(loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.12))
plt.tight_layout()
plt.show()

*Figure: seven policies over five years on three measures: the whole list (left), patients waiting more than 12 months (middle) and cumulative extra spending (right).*

A single number per policy hides the shape over time, but a table helps the comparison. **Long-wait months** adds up, month by month, the number of patients over 12 months: it counts harm that a snapshot at the end would miss. The last column divides the extra spending by the long-wait months avoided compared with doing nothing.

In [ ]:
def summary(d, ref):
    dt = d.index[1] - d.index[0]
    lw = d["Waiting over 12 months"].sum() * dt
    lw_ref = ref["Waiting over 12 months"].sum() * dt
    cost = d["Extra cost"].iloc[-1]
    avoided = lw_ref - lw
    return {"list, end 2028": d["Waiting list"].iloc[-1],
            "over 12 m, end 2028": d["Waiting over 12 months"].iloc[-1],
            "over 12 m, lowest": d["Waiting over 12 months"].min(),
            "long-wait months (k)": lw / 1000,
            "cost (EUR m)": cost,
            "EUR per long-wait month avoided":
                cost * 1e6 / avoided if avoided > 0 else np.nan}


table = pd.DataFrame({n: summary(d, runs["Do nothing"])
                      for n, d in runs.items()}).T
print(table.round(1).to_string())

Four readings stand out. The permanent increase ends best of the single measures on the list (63,581) and on long waiters (27,394). The initiative costs most (EUR 23.6 million) for a result that is fading by 2028. Treating the longest waiters first costs nothing at all, and of the measures that cost money validation is by far the cheapest per long-wait month avoided (about EUR 6), because a phone call or a letter costs little; but it is a one-off. The package, three measures together, cuts the long waiters of doing nothing by more than half (16,260 against 38,780) for EUR 19.4 million.

**Temporary against permanent.** The initiative brings waits down faster. The list bottoms out at 57,624 when it ends in early 2025, and the number over 12 months goes on falling for another year, to 25,437 in early 2026, a level the permanent increase never reaches. Then the gap between referrals and treatments reopens and both climb again. The permanent increase works more slowly but keeps working: by the end of 2028 it is ahead on the list (63,581 against 66,632) and on long waiters (27,394 against 28,932), and costs EUR 5.8 million less. The initiative still wins on long-wait months added up over the five years (1.68 million against 1.84 million), because it helped earlier. Which matters more is a judgement for the decision maker, and the table makes it visible. This is the bathtub of chapter 1 in policy form: a one-off pulse of extra outflow changes the level once; only a change in the rates changes where the level is heading.

**Targeting and validation.** Treating the longest waiters first changes nobody's total: the list ends exactly where it would with no policy. It moves the waiting from the long band to the shorter ones, which matters to the patients in it and to any target written in terms of long waits. Validation removes patients once, cheaply, and the list then grows again from its lower level.

**Try it 2.**

- *Check.* Which policy leaves the total list at the end of 2028 exactly where doing nothing leaves it, and why must it?
- *Apply.* Run the initiative for 24 months at 10% instead of 12 months at 20%. Does the same amount of extra capacity, spread out, do better or worse at the end of 2028 on long waiters?
- *Extend.* Add a seventh measure: how far the number over 12 months has risen from its lowest point by the end of 2028 (a "relapse" measure). Which policies relapse?

<details><summary>Worked answer</summary>

*Check.* Longest waiters first. Changing the weights changes who is treated, not how many: treatments still equal capacity, so referrals minus treatments, and with it the total, is untouched. The list ends at 79,735 in both runs.

*Apply.* Almost the same: 28,838 waiting over 12 months at the end of 2028 with 10% for 24 months against 28,932 with 20% for 12 months, at EUR 23.8 against 23.6 million. In this model what counts is the total number of extra treatments, not how they are spread. (In practice a longer, smaller initiative may be easier to staff.)

*Extend.* Measured as the rise from the lowest point to the end of 2028, doing nothing relapses by 4,118, the initiative by 3,495, validation by 3,854 and targeting by 3,385. The permanent increase, demand management and the package are still falling at the end: they change the rates, so they do not relapse within the five years.

</details>

In [ ]:
spread = pm.run(temporary_extra=0.10, temporary_duration=24)
short = runs["Initiative +20%, 1 year"]
print("Over 12 months at end 2028: 20% for 12 months",
      round(short["Waiting over 12 months"].iloc[-1]),
      "| 10% for 24 months", round(spread["Waiting over 12 months"].iloc[-1]))
print("Cost:", round(short["Extra cost"].iloc[-1], 1),
      "against", round(spread["Extra cost"].iloc[-1], 1), "EUR million")
relapse = {n: round(d["Waiting over 12 months"].iloc[-1]
                    - d["Waiting over 12 months"].min())
           for n, d in runs.items()}
print("Rise from the lowest point to end 2028:", relapse)

## 3. Will the ranking survive what we do not know?

Every number in section 2 rests on guesses. The two that matter most are how fast referrals and capacity grow, because their difference decides whether the bath fills or drains. A **scenario** is one combination of such guesses; a **robust** policy is one that does well across all plausible scenarios, not only in the one we happen to believe.

We cross three referral growth rates (1%, 3% and 5% a year) with three capacity growth rates (1%, 2% and 3% a year). Nine scenarios times seven policies is 63 runs. Because every policy is a constant, `run_batch` does all of them in one call.

In [ ]:
grid = [{"referral_growth": rg, "capacity_growth": cg}
        for rg in (0.01, 0.03, 0.05) for cg in (0.01, 0.02, 0.03)]
rows, labels = [], []
for s, scen in enumerate(grid):
    for name, p in POLICIES.items():
        rows.append({**scen, **p})
        labels.append((name, f"ref {scen['referral_growth']:.0%}, "
                             f"cap {scen['capacity_growth']:.0%}"))
settings = pd.DataFrame(rows).fillna(pd.Series(pm.params))
out = pm.run_batch(settings, include=["Waiting over 12 months",
                                      "Waiting list", "Extra cost"])
end = out[out.time == out.time.max()].pivot(index="run", columns="variable",
                                            values="value")
end.index = pd.MultiIndex.from_tuples([labels[i] for i in end.index],
                                      names=["policy", "scenario"])
long_end = end["Waiting over 12 months"].unstack("scenario")
print(f"{len(settings)} runs. Over 12 months at the end of 2028 (thousand):")
print((long_end / 1000).round(1).to_string())

The spread is wide. Doing nothing ends anywhere between 23,400 (referrals grow 1%, capacity 3% a year) and 55,300 (referrals 5%, capacity 1%) patients over 12 months. The package ends between 5,300 and 30,600. The future we do not know matters more than the choice between most single measures: within any one column the six single measures (doing nothing aside) end within about 6,500 of each other, while across columns the same policy moves by up to 30,000.

A **regret** is how much worse a policy does in a scenario than the best policy would have done in that same scenario. It answers the question a board fears most: if we choose this and the world turns out like that, how much will we wish we had chosen differently? The policy with the smallest worst-case regret is the **minimax-regret** choice. The package is in a different class (it is three measures and costs more), so the regret table compares the six single measures, the choice when there is money for only one.

In [ ]:
single = long_end.drop(index="Package")
regret = single - single.min()
summary_r = pd.DataFrame({
    "mean over 12 m (k)": single.mean(axis=1) / 1000,
    "worst over 12 m (k)": single.max(axis=1) / 1000,
    "max regret (k)": regret.max(axis=1) / 1000,
    "times best": (regret == 0).sum(axis=1),
    "mean cost (EUR m)": end["Extra cost"].unstack("scenario")
    .drop(index="Package").mean(axis=1),
}).sort_values("max regret (k)")
print(summary_r.round(1).to_string())

The permanent increase is the best single measure in all nine scenarios, so its regret is zero everywhere and it is the minimax-regret choice. The initiative is never more than 1,800 patients behind it, but costs more. Two cheap measures are worth noticing: treating the longest waiters first costs nothing and does as well on long waits as demand management (a mean of 31,400 each), which costs EUR 17.1 million. Robustness here confirms the central-case ranking; it will not always, and that is the point of checking.

**Try it 3.**

- *Check.* In how many of the nine scenarios does doing nothing leave more than 40,000 patients waiting over 12 months at the end of 2028?
- *Apply.* Judge regret on the whole list at the end of 2028 instead of long waiters. Does the minimax-regret choice change? Explain why in one sentence.
- *Extend.* Add a third uncertain parameter, the share of the list that validation removes (5%, 10% or 15%), so the grid has 27 scenarios. Does validation become anyone's best choice?

<details><summary>Worked answer</summary>

*Check.* Four of the nine: every scenario with referrals growing at 5% a year, and the one with 3% referral growth and 1% capacity growth.

*Apply.* No: the permanent increase still has zero regret in every scenario. The big change is targeting, whose worst-case regret jumps to 16,600, the same as doing nothing, because it moves patients between bands without changing the total.

*Extend.* The permanent increase is the best single measure in all 27 scenarios. Even when validation removes 15% of the list, the one-off cut is regained before 2028, while the permanent increase keeps working.

</details>

In [ ]:
print("Do nothing above 40,000:",
      int((long_end.loc["Do nothing"] > 40000).sum()), "of 9")
lst = end["Waiting list"].unstack("scenario").drop(index="Package")
print("Max regret on the whole list (thousand):")
print(((lst - lst.min()).max(axis=1) / 1000).round(1).sort_values()
      .to_string())
rows27 = []
for scen in grid:
    for vf in (0.05, 0.10, 0.15):
        for name, p in POLICIES.items():
            if name == "Package":
                continue
            q = dict(p)
            if "validation_fraction" in q:
                q["validation_fraction"] = vf
            rows27.append({**scen, **q, "policy": name, "vf": vf})
s27 = pd.DataFrame(rows27)
o27 = pm.run_batch(s27.drop(columns=["policy", "vf"])
                   .fillna(pd.Series(pm.params)),
                   include=["Waiting over 12 months"])
e27 = o27[o27.time == o27.time.max()].set_index("run")["value"]
s27["over12"] = e27.values
s27["scenario"] = (s27.referral_growth.astype(str) + "/"
                   + s27.capacity_growth.astype(str) + "/"
                   + s27.vf.astype(str))
best = s27.loc[s27.groupby("scenario")["over12"].idxmin(), "policy"]
print("Best single measure across 27 scenarios:",
      best.value_counts().to_dict())

## 4. How long until the backlog is cleared?

Boards ask "when will it be fixed?" more often than any other question. Two small molecules show how to answer it and how the usual answer goes wrong.

**Estimated Remaining Duration** (`estRemaining`) estimates the time left on a job as the work still to do divided by the rate at which we expect to do it:

$$\text{Estimated remaining duration} = \frac{\text{Work to do}}{\max(1,\ \text{Anticipated rate of accomplishing work})}.$$

The work itself is done at the actual rate, limited so the stock cannot go below zero. The estimate is only as good as the anticipated rate. After Hines, *Molecules of Structure* [@hines_molecules]; ours guards the division with `MAX(1, ...)` and adds a minimum task time of 0.5 week to the outflow, and its example is a painting job measured in square feet.

In [ ]:
er = lb.molecule("estRemaining")
print(er.equations())
d = er.run(preset="Slower than anticipated")
done_at = d.index[d["Work to do"] < 1][0]
print(f"\nEstimate at week 0: {d.loc[0, 'Estimated remaining duration']:.1f}"
      f" weeks. The work is actually finished in week {done_at:.1f}.")
for t in (0, 10, 20):
    est = d.loc[t, "Estimated remaining duration"]
    print(f"week {t:2d}: estimate {est:5.1f} weeks left, "
          f"really {done_at - t:5.1f}")

At week 0 the estimate says 24 weeks; the job actually takes 31.6. The estimate falls by 0.8 week each week, not by 1, because the work goes at 200 sq ft/week rather than the 250 anticipated. It is always too low, and the error shrinks only as the job nears its end (8.0 weeks estimated against 11.6 real at week 20).

**Estimated Completion** (`estCompletion`) adds the twist that matters for waiting lists: work keeps arriving while you work. Work remaining has an inflow, `Adding scope`, as well as the outflow `Accomplishing`:

$$\frac{d\,\text{Work remaining}}{dt} = \text{Scope creep} - \min\left(\text{Work rate},\ \frac{\text{Work remaining}}{0.5}\right).$$

The naive estimate of the finish date divides what is left by the work rate and adds it to today. With no scope creep it is right; with scope creep almost as fast as the work, the date recedes week after week. After Hines, *Molecules of Structure* [@hines_molecules]; ours has no estimate variable inside the model, so we compute the estimated completion date from the run, and the inflow is a constant scope creep.

In [ ]:
ec = lb.molecule("estCompletion")
fig, ax = plt.subplots()
for preset in ("Fixed scope (finishes)", "Scope creep (recedes)"):
    d = ec.run(preset=preset, T=60)
    rate = ec.preset(preset)["workRate"]
    naive = d.index + d["Work remaining"] / rate     # today + left / rate
    ax.plot(d.index, naive, label=preset)
    print(f"{preset}: naive finish at week 0 is {naive.iloc[0]:.1f}, "
          f"at week 30 it is {naive[30.0]:.1f}; work left at week 60 "
          f"{d['Work remaining'].iloc[-1]:.0f}")
ax.plot([0, 60], [0, 60], color="grey", lw=1, label="today")
ax.set(title="The naive finish date recedes when work keeps arriving",
       xlabel="week the estimate is made",
       ylabel="estimated finish (week)")
ax.legend()
plt.show()

*Figure: the naive estimated finish date (today plus work left divided by work rate), made each week. With a fixed scope the estimate stays put; with scope creep it climbs almost parallel to today's date.*

With a fixed scope the estimate made at week 0 (16.7 weeks) is right. With scope creep of 28 a week against a work rate of 30, the same week-0 estimate of 16.7 weeks has become 44.7 by week 30: in 30 weeks the finish moved 28 weeks further away, and at week 60, 380 of the 500 units are still to do.

The cure is in the denominator: divide by the **net** rate, the work rate minus the rate at which new work arrives. For a waiting list, the backlog is the list above where it would settle, and the net rate is capacity minus referrals. If that net rate is shrinking, because referrals grow faster than capacity, even the net estimate is optimistic. Here is the over-12 band under the package policy: we estimate, at several points in time, how long the band would take to empty, and compare with the run.

In [ ]:
pk = pm.run(**POLICIES["Package"], T=120, include="all")
lw = pk["Waiting over 12 months"]
net_out = (pk["Treating 4"] + pk["Validating 4"]) - pk["Ageing 3"]
rows = []
for t in (12, 24, 36, 60):
    naive = lw[t] / (pk.loc[t, "Treating 4"] + pk.loc[t, "Validating 4"])
    net = lw[t] / net_out[t] if net_out[t] > 0 else np.inf
    rows.append({"month": t, "over 12 m": lw[t],
                 "naive months left": naive, "net months left": net})
print(pd.DataFrame(rows).round(1).to_string(index=False))
print(f"Lowest over-12 count in 10 years: {lw.min():.0f} "
      f"at month {lw.idxmin():.0f}")

At month 12 the long-wait band holds 22,624 patients. Dividing by the rate at which they leave says 7.6 months, which is the naive estimate, and badly wrong: patients keep ageing into the band. Dividing by the net rate says 51 months. But a year later the net estimate has grown to 123 months, and by month 60 to 483. The band never empties: over ten years it is lowest at 16,067, in month 71, and then rises, because referrals grow faster than capacity and the net clearance rate shrinks towards zero. Tell a board the net estimate, and say that it holds only if the net rate holds.

**Try it 4.**

- *Check.* In `estCompletion` with a work rate of 30 and scope creep of 28 per week, how many weeks until the 500 units of work are down to 20? Work it out on paper first, then run it. Does the work ever reach zero?
- *Apply.* For the whole list under "Permanent +5%", compute the net months to clear the list down to 40,000 patients at month 12, using capacity minus referrals at that month. Then find in a run when (if ever) it gets there.
- *Extend.* Write a function `months_to_clear(run, target)` that returns the first month the list is at or below `target`, or `None`. Use it to find the smallest permanent capacity increase (in steps of 1%) that brings the list below 40,000 within five years.

<details><summary>Worked answer</summary>

*Check.* The net rate is 30 minus 28, or 2 units a week, so 480 units take 240 weeks; the run gives 240.1. It never reaches zero: as the stock gets small, `Work_remaining / 0.5` limits the outflow, and the stock settles at 14 units, where that limit equals the scope creep of 28 a week.

*Apply.* At month 12 the list is 28,033 above 40,000 and capacity exceeds referrals by 200 patients/month, so the net estimate is 140 months. The run never gets there, even in 20 years: referrals grow faster than capacity, so the net rate shrinks and then turns negative. The estimate was the receding finish of `estCompletion`.

*Extend.* A permanent increase of 13% is the smallest whole percent that brings the list to 40,000 within five years (by month 55).

</details>

In [ ]:
d = ec.run(preset="Scope creep (recedes)", T=400)
print("Scope creep: work left is down to 20 in week",
      d.index[d["Work remaining"] <= 20][0], "| at week 400:",
      round(d["Work remaining"].iloc[-1], 1))
p5 = pm.run(**POLICIES["Permanent +5%"], T=240, include="all")
gap = p5.loc[12, "Waiting list"] - 40000
net = p5.loc[12, "Capacity"] - p5.loc[12, "Referrals"]
print(f"Month 12: {gap:.0f} above 40,000, net rate {net:.0f}/month, "
      f"estimate {gap / net:.0f} months")


def months_to_clear(run, target):
    hit = run.index[run["Waiting list"] <= target]
    return float(hit[0]) if len(hit) else None


print("Permanent +5%, months to 40,000:", months_to_clear(p5, 40000))
for pct in range(1, 21):
    r = pm.run(permanent_extra=pct / 100, include=["Waiting list"])
    when = months_to_clear(r, 40000)
    if when is not None:
        print(f"Smallest increase: {pct}% (list at 40,000 by month {when})")
        break

## 5. A one-page brief

A decision maker reads one page, often in two minutes, and does not read code. The brief has one figure and five plain sentences: what is happening, what we recommend, what it buys, what it costs, and how sure we are. Every number in it is pulled from the runs, so the brief is rebuilt whenever the model changes and can never drift from it.

In [ ]:
def band(policy):
    # lowest and highest over-12 count across the nine scenarios, by month
    idx = [i for i, (p, _) in enumerate(labels) if p == policy]
    sub = out[(out.variable == "Waiting over 12 months") & out.run.isin(idx)]
    piv = sub.pivot(index="time", columns="run", values="value")
    return piv.min(axis=1), piv.max(axis=1)


best_single = summary_r.index[0]
show = ["Do nothing", best_single, "Package"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in show:
    lo, hi = band(name)
    line, = ax.plot(year(runs[name].index),
                    runs[name]["Waiting over 12 months"] / 1000, label=name)
    ax.fill_between(year(lo.index), lo / 1000, hi / 1000, alpha=0.15,
                    color=line.get_color())
ax.set(title="Patients waiting over 12 months, 2024 to 2028",
       xlabel="year", ylabel="patients (thousand)", ylim=(0, None))
ax.legend(title="lines: central case; shading: range of 9 scenarios")
plt.show()

nothing, pkg = runs["Do nothing"], runs["Package"]
k = lambda x: f"{x / 1000:,.0f},000"
wins = summary_r.loc[best_single, "times best"]
n_best = "all of" if wins == len(grid) else f"{wins} of"
brief = [
    f"About {k(last['w12_plus'])} patients have waited more than a year, "
    f"and if nothing changes that will be about "
    f"{k(nothing['Waiting over 12 months'].iloc[-1])} by the end of 2028.",
    f"We recommend a package of three measures: "
    f"{POLICIES['Package']['permanent_extra']:.0%} more permanent "
    f"capacity, a check of who still needs care, and seeing the longest "
    f"waiters first.",
    f"Together they would bring long waits down to about "
    f"{k(pkg['Waiting over 12 months'].iloc[-1])} by the end of 2028, "
    f"and the whole list from {k(last['total'])} to about "
    f"{k(pkg['Waiting list'].iloc[-1])}.",
    f"The extra cost is about EUR {pkg['Extra cost'].iloc[-1]:.0f} million "
    f"over five years; a one-year blitz of the same size would cost more "
    f"and its gains fade once it ends.",
    f"If only one measure is affordable, permanent capacity is the safest "
    f"choice: it did best of the single measures in {n_best} the nine "
    f"futures we tested.",
]
for i, s in enumerate(brief, 1):
    print(f"{i}. {s}")

*Figure: the brief's figure. Lines are the central case, shading the range across the nine growth scenarios of section 3.*

The five sentences come straight from the runs. The figure answers the follow-up question before it is asked: even in the worst of the nine futures, the package ends below the central case of doing nothing.

Notice what the brief leaves out: the model, the equations, the word "scenario grid". It keeps the numbers rounded to what the model can support, it names the cost, and its last sentence carries the uncertainty in plain words. When the board asks "how sure are you?", the shaded range is the answer.

**Try it 5.**

- *Check.* Count the numbers in the five sentences. Can each be traced to one line of code above?
- *Apply.* Rewrite sentence 5 for a board that cares most about the total list rather than long waiters, using the regret on the whole list from section 3.
- *Extend.* Add a sixth sentence, used only when the package does not get the over-12 count below 20,000 by the end of 2028 in the worst scenario, warning that more is needed.

<details><summary>Worked answer</summary>

*Check.* Yes. The 35,000 is `last["w12_plus"]`, the 39,000 and 16,000 come from the over-12 column of the "Do nothing" and "Package" runs, 70,000 and 57,000 from the list, the EUR 19 million from `Extra cost`, and the nine from the scenario grid.

*Apply.* For example: "If only one measure is affordable and the whole list is what matters, permanent capacity is still the safest choice; seeing the longest waiters first would cost nothing but would not shorten the list at all." The cell below confirms that the permanent increase has the lowest worst-case regret on the whole list.

*Extend.* In the worst scenario the package leaves 30,592 waiting over 12 months, so the sixth sentence is triggered: "In the least favourable future we tested, long waits stay above 31,000, so more will be needed."

</details>

In [ ]:
lst_regret = (lst - lst.min()).max(axis=1).sort_values()
print("Lowest worst-case regret on the whole list:", lst_regret.index[0])
worst_pkg = long_end.loc["Package"].max()
print(f"Package, worst scenario, over 12 m at end 2028: {worst_pkg:.0f}")
if worst_pkg > 20000:
    print("6. In the least favourable future we tested, long waits stay "
          f"above {k(worst_pkg)}, so more will be needed.")

## 6. A simulator the board can play with

People trust a model more when they can move its levers themselves and see the result. The first step towards an interactive simulator is a plain function: inputs in the units the user thinks in (percent, months), the model run inside, and a figure and a few numbers out. Everything else is decoration.

In [ ]:
def simulator(extra_capacity_pct=0, initiative_pct=0, initiative_months=12,
              demand_cut_pct=0, validation_pct=0, long_wait_weight=0.8,
              referral_growth_pct=3, show=True):
    # one run of the policy model from the settings a user would choose
    d = pm.run(permanent_extra=extra_capacity_pct / 100,
               temporary_extra=initiative_pct / 100,
               temporary_duration=initiative_months,
               demand_reduction=demand_cut_pct / 100,
               validation_fraction=validation_pct / 100,
               weight_4=long_wait_weight,
               referral_growth=referral_growth_pct / 100)
    result = {"list end 2028": round(d["Waiting list"].iloc[-1]),
              "over 12 m end 2028":
                  round(d["Waiting over 12 months"].iloc[-1]),
              "cost EUR m": round(float(d["Extra cost"].iloc[-1]), 1)}
    if show:
        fig, ax = plt.subplots(figsize=(7, 3.5))
        ax.plot(year(base.index), base["Waiting list"] / 1000,
                label="list, no policy")
        ax.plot(year(d.index), d["Waiting list"] / 1000, label="list")
        ax.plot(year(d.index), d["Waiting over 12 months"] / 1000,
                label="over 12 months")
        ax.set(title="Simulator", xlabel="year",
               ylabel="patients (thousand)", ylim=(0, None))
        ax.legend()
        plt.show()
    return result


simulator(extra_capacity_pct=8, long_wait_weight=2)

*Figure: one call of the simulator, with 8% more permanent capacity and the longest waiters seen first, against the no-policy list.*

With 8% more permanent capacity and a weight of 2 on the longest waiters, the list ends 2028 at 53,889, of whom 14,652 have waited over 12 months, for EUR 28.4 million.

With the function in place, sliders are one line. The optional `ipywidgets` package (not needed anywhere else in this book, and not installed here) turns each keyword into a slider in Jupyter or Colab:

```python
from ipywidgets import interact     # pip install ipywidgets
interact(simulator, extra_capacity_pct=(0, 20, 1), initiative_pct=(0, 40, 5),
         initiative_months=(3, 24, 3), demand_cut_pct=(0, 10, 1),
         validation_pct=(0, 20, 1), long_wait_weight=(0.5, 4, 0.1),
         referral_growth_pct=(0, 6, 0.5), show=True)
```

The same function can sit behind a web page (for instance with a dashboard library) with no change to the model. Two habits make such a simulator honest: show the no-policy run on every chart, so every lever is judged against a baseline, and let users move the uncertain assumptions (here `referral_growth_pct`) as well as the policies, so they see for themselves how much depends on them.

**Try it 6.**

- *Check.* Call `simulator()` with no arguments. Do its numbers match the "Do nothing" row of the table in section 2?
- *Apply.* With `show=False`, use the simulator in a loop to find the smallest `extra_capacity_pct` (whole numbers) for which the list at the end of 2028 is below the list at the start.
- *Extend.* Add an argument `start_month` for when policies begin, and show that starting a 5% permanent increase a year later costs less but leaves more long waiters at the end of 2028.

<details><summary>Worked answer</summary>

*Check.* Yes: 79,735 on the list, 38,780 over 12 months and no cost.

*Apply.* 4%: the smallest whole-percent permanent increase for which the list at the end of 2028 is below the 69,927 it started at.

*Extend.* `policy_start` is already a constant of the model, so the new argument passes straight through as `policy_start=start_month`. Starting 5% a year later (month 15 instead of 3) saves EUR 3.6 million (14.2 against 17.8) but leaves 29,760 waiting over 12 months at the end of 2028 instead of 27,394.

</details>

In [ ]:
print(simulator(show=False))
for pct in range(0, 21):
    if simulator(extra_capacity_pct=pct, show=False)["list end 2028"] \
            < last["total"]:
        print("Smallest whole-percent increase that shrinks the list:", pct)
        break
for start in (3, 15):
    d = pm.run(permanent_extra=0.05, policy_start=start)
    lw_end = d["Waiting over 12 months"].iloc[-1]
    print(f"start month {start}: over 12 m {lw_end:.0f}, "
          f"cost {d['Extra cost'].iloc[-1]:.1f} EUR m")

## 7. Replicating a published result

A new implementation earns trust when it reproduces a result someone else published. Replication checks the engine and your understanding together, and it is a habit worth carrying into every project that builds on a published model. We take one of the best known: Bass's model of how a new product spreads [@bass1969]. His qualitative result is that adoptions per period rise to a single peak and then fall, so that cumulative adoption traces an S, and that the peak comes at

$$t^{*} = \frac{\ln(q/p)}{p + q},\qquad \text{with } \frac{\text{Adopters}(t^{*})}{N} = \frac12 - \frac{p}{2q},$$

when imitation $q$ exceeds innovation $p$ (both fractions per unit time). If $q \le p$ there is no interior peak: adoptions are highest at the start.

The **Diffusion** molecule (`diffusion`) from chapter 3 is our implementation, with `Innovation p` and `Imitation q` in percent per week and a population of 1,000. We check the timing and the height of the peak against the formulas, at the molecule's time step of 0.1 week and at a tenth of it.

In [ ]:
dif = lb.molecule("diffusion")
rows = []
for p, q in [(1, 18), (0.5, 35), (4, 5), (4, 2)]:
    pf, qf = p / 100, q / 100
    t_star = np.log(qf / pf) / (pf + qf) if qf > pf else 0.0
    f_star = 0.5 - pf / (2 * qf) if qf > pf else 0.0
    row = {"p %": p, "q %": q, "Bass t* (week)": t_star,
           "Bass share at t*": f_star}
    for dt in (0.1, 0.01):
        d = dif.run(T=80, dt=dt, innovation_p=p, imitation_q=q)
        t_pk = d["Adoption"].idxmax()
        row[f"model t*, dt {dt}"] = t_pk
        row[f"model share, dt {dt}"] = d.loc[t_pk, "Adopters"] / 1000
    rows.append(row)
rep = pd.DataFrame(rows)
print(rep.round(3).to_string(index=False))

Our molecule reproduces the published result. With $p$ 1% and $q$ 18% per week, Bass's formula puts the peak at 15.21 weeks with 47.2% adopted; the molecule gives 15.3 weeks at `dt` 0.1 and 15.22 at `dt` 0.01, with the same share. The other cases agree as closely, and with $p$ 4% and $q$ 2% the adoption rate is highest at time 0, as the theory says it must be.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for p, q in [(1, 18), (0.5, 35), (4, 5), (4, 2)]:
    d = dif.run(T=80, dt=0.01, innovation_p=p, imitation_q=q)
    ax1.plot(d.index, d["Adoption"], label=f"p {p}%, q {q}%")
    ax2.plot(d.index, d["Adopters"])
ax1.set(title="Adoptions per week", xlabel="week",
        ylabel="adoptions (people/week)")
ax2.set(title="Cumulative adopters", xlabel="week", ylabel="people")
ax1.legend()
plt.tight_layout()
plt.show()

*Figure: the replication. Adoptions per week rise to one peak and fall whenever imitation exceeds innovation (left); cumulative adoption traces an S (right). With p 4% and q 2% per week, adoptions are highest at the start.*

The replication succeeds on both counts: the shape (one peak, an S, no peak when $q \le p$) and the numbers (the peak time and share match Bass's formulas once the time step is small). A good replication report says what was reproduced, by how much it differs, and why: here the small differences at `dt` 0.1 are the Euler error of chapter 1, and they shrink when the step shrinks.

**Try it 7.**

- *Check.* With $p$ 1% and $q$ 18% per week, compute $t^*$ by hand. Does it match the table?
- *Apply.* Bass's formula says the peak time depends on $p$ and $q$, not on the population. Run the molecule with a population of 10,000 (set `Potential` to start at 10,000 with `with_init`) and confirm.
- *Extend.* Replicate a second classic: Forrester's finding that a stock manager who ignores the supply line makes inventory oscillate [@forrester1961]. Use the Stock Management molecule (`stockmgmt`) from chapter 6 with `supply_line_weight` 0 and 1, and report the peak inventory and the number of times inventory crosses its target in 100 weeks.

<details><summary>Worked answer</summary>

*Check.* $\ln(18/1)/(0.01 + 0.18) = 2.890/0.19 = 15.21$ weeks, the value in the table.

*Apply.* With 10,000 people the peak is still in week 15.22 and the share at the peak still 0.472: the timing does not depend on the size of the population.

*Extend.* With the *Long lead time* preset, a supply line weight of 0 sends inventory up to 116 against a target of 100, crossing the target 4 times in 100 weeks; a weight of 1 brings it up to 100 without ever crossing. The qualitative result, oscillation from ignoring the supply line, is reproduced.

</details>

In [ ]:
print("t* =", round(np.log(18 / 1) / 0.19, 2), "weeks")
big = dif.with_init("Potential", 10000).run(T=80, dt=0.01)
print("Population 10,000: peak in week", big["Adoption"].idxmax(),
      "share", round(big.loc[big["Adoption"].idxmax(), "Adopters"] / 10000, 3))
sm = lb.molecule("stockmgmt")
for w in (0, 1):
    d = sm.run(T=100, supply_line_weight=w, preset="Long lead time")
    inv = d["Inventory"]
    above = (inv > 100).astype(int)
    print(f"supply line weight {w}: peak inventory {inv.max():.0f}, "
          f"crossings of 100: {int(above.diff().abs().sum())}")

## Check it

Before any policy result leaves your desk, run three checks.

1. **Units and money.** Every flow that feeds `Extra cost` is in EUR per month divided by a million; a month of 5% extra capacity at EUR 1,100 per patient should add about EUR 0.3 million. Check one month by hand.
2. **The no-policy run is the baseline.** Every policy setting at its "off" value must reproduce the no-policy run exactly; otherwise a policy is leaking into the baseline.
3. **Time step.** Halve `dt` and confirm the ranking of the policies does not change.

In [ ]:
d = pm.run(permanent_extra=0.05)
print("Spending in month 4 (EUR million/month):",
      round(d.loc[4, "Spending"], 3), "| by hand:",
      round(d.loc[4, "Capacity trend"] * 0.05 * 1100 / 1e6, 3))
off = pm.run(permanent_extra=0, temporary_extra=0, demand_reduction=0,
             validation_fraction=0)
print("Off settings reproduce the baseline:",
      np.allclose(off["Waiting list"], base["Waiting list"]))
rank = {}
for dt in (0.25, 0.125):
    ends = {n: pm.run(dt=dt, **p)["Waiting over 12 months"].iloc[-1]
            for n, p in POLICIES.items()}
    rank[dt] = sorted(ends, key=ends.get)
print("Same ranking at dt 0.125:", rank[0.25] == rank[0.125])

## Running case

This chapter's increment turns the waiting-list model into a policy model. The bands of the data become an ageing chain; treatments are shared by priority weights; six policies (permanent capacity, a temporary initiative, demand management, validation, targeting the longest waiters, and a package) are constants that are zero when off; and a cost stock adds up what each policy spends. The model is judged on the list, on long waiters and on cost, across a grid of nine growth scenarios, and the results feed a one-page brief and a simulator function.

## Tell the decision maker

Use the five sentences of section 5 as they stand, with the figure. If you are asked for one more sentence, make it the comparison that surprises people most: a one-year waiting-list initiative and a modest permanent increase buy about the same number of extra treatments, but the initiative costs about a third more and only the permanent increase still helps after the money stops.

## Going further

- **Optimisation.** Instead of comparing a handful of policies, search for the best combination under a budget: a grid over two or three levers (as in chapter 10's calibration search) or a simple Nelder-Mead on a score that adds long-wait months and cost at a price per month of waiting.
- **Deep uncertainty.** With many uncertain parameters, sample hundreds of scenarios with `run_batch` and look for the conditions under which a policy fails, rather than its average. This is the idea behind robust decision making and exploratory modelling.
- **Interfaces.** The simulator function of section 6 can go behind `ipywidgets` sliders in a notebook or behind a small web dashboard. Keep the model in one place so the interface cannot drift from it.
- **Replication libraries.** Published system dynamics models in Vensim or XMILE format can be imported to Python with PySD [@pysd2022] and checked against your own implementation, as in section 7; chapter 12 returns to this.

## Mini-project

The board adds a target: nobody should wait more than 12 months by the end of 2027. Find the cheapest combination of the six measures (in steps of 1% for capacity and demand, 5% for validation and 0.5 for the weight) that meets it in the central case, then test it on the nine scenarios of section 3. Write the result as a one-page brief with one figure and five sentences, and say in the last sentence in how many of the nine futures your plan meets the target.

## Further reading

Policy analysis, robustness and communication with system dynamics models are treated at length in @sterman2000. On building confidence before using a model for policy, see @barlas1996 and the best-practice survey of @martinezmoyano2013. For health-care applications and their audiences, see @homer2006. The diffusion model replicated here is @bass1969; the oscillation result in the last exercise goes back to @forrester1961. If you want to play with the two home molecules of this chapter, the LoopBricks app has them at [Estimated Remaining Duration](https://waelrash1.github.io/loopbricks/#/m/estRemaining) and [Estimated Completion](https://waelrash1.github.io/loopbricks/#/m/estCompletion).